# CRISPRa vs. CRISPRi Screen Comparison Plots

This notebook makes the figures in plotting/ that compare the CRISPRa and CRISPRi screens (mostly QC checks).

## Load modules

In [1]:
!module load scipy-stack/2026a
!module list


Currently Loaded Modules:
  1) CCconfig                 16) calibre/8.6.0
  2) gentoo/2023         (S)  17) libreqda/1.1.4
  3) gcccore/.12.3       (H)  18) flexiblascore/.3.3.1         (H)
  4) gcc/12.3            (t)  19) java/17 -> java/17.0.6       (t)
  5) hwloc/2.9.1              20) r/4.4.0                      (t)
  6) ucx/1.14.1               21) rstudio-server/4.4           (t)
  7) libfabric/1.18.0         22) python/3.11 -> python/3.11.5 (t)
  8) pmix/4.2.4               23) ipykernel/2026a
  9) ucc/1.2.0                24) ipython-kernel/3.11          (S)
 10) openmpi/4.1.5       (m)  25) openrefine/3.9.3
 11) flexiblas/3.3.1          26) mlflow/3.8.1
 12) blis/0.9.0               27) tensorboard/2.20.0
 13) StdEnv/2023         (S)  28) jupyterlab-apps/1.0
 14) mii/1.1.2                29) scipy-stack/2026a            (math)
 15) code-server/4.101.2

  Where:
   S:     Module is Sticky, requires --force to unload or purge
   m:     MPI implementations / Implémentations MP

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import pearsonr, spearmanr

An sgRNA is called significant using the same criteria as the individual screen notebooks: FDR < 0.05 and a log2 fold change outside the 95% reference interval of the non-targeting sgRNAs (mean ± 1.96 SD). The interval bounds below are copied from the output of each screen.

In [3]:
CSV = "crisprA_crisprI_rnaseq.csv"
FDR = 0.05

#Non-targeting sgRNA 95% reference intervals (log2 fold change) for each screen
CI_A = {"low_flz": (-1.5968, 0.9892), "high_flz": (-1.4210, 0.9810)}
CI_I = {"low_flz": (-2.8800, 1.0267), "high_flz": (-3.0950, 1.6368)}

df_all = pd.read_csv(CSV)
df_all["Gene"] = df_all["Gene"].astype(str)

#An sgRNA is significant if FDR < 0.05 and its log2FC is outside the non-targeting interval
def is_significant(fc, p, ci):
    lo, hi = ci
    return (p < FDR) & ((fc < lo) | (fc > hi))

Compares the log2 fold changes of each sgRNA between the two FLZ concentrations, for each screen. Non-targeting sgRNAs are removed. Red points are sgRNAs that are significant in at least one FLZ condition, and the Pearson and Spearman correlations are calculated using all sgRNAs.

In [4]:
OUT_DIR = "./plotting/flz_low_vs_high"
os.makedirs(OUT_DIR, exist_ok=True)

#Remove the non-targeting sgRNAs
df = df_all[~df_all["Gene"].str.lower().str.startswith("non-targeting")].copy()

def plot_low_vs_high(suffix, ci, title, out_prefix):
    xcol   = f"log2_fold_change_low_flz_{suffix}"
    ycol   = f"log2_fold_change_high_flz_{suffix}"
    p_low  = f"adj_p_value_low_flz_{suffix}"
    p_high = f"adj_p_value_high_flz_{suffix}"

    #Keep only sgRNAs with values in both FLZ conditions for this screen
    sub = df[["Alias", "Gene", xcol, ycol, p_low, p_high]].dropna().copy()

    #Significant in either FLZ condition
    sig_low  = is_significant(sub[xcol], sub[p_low],  ci["low_flz"])
    sig_high = is_significant(sub[ycol], sub[p_high], ci["high_flz"])
    sub["sig_either"] = sig_low | sig_high

    #Correlations across all sgRNAs
    x = sub[xcol].to_numpy()
    y = sub[ycol].to_numpy()
    pr, _ = pearsonr(x, y)
    sr, _ = spearmanr(x, y)

    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    ax.set_box_aspect(1)

    #Non-significant sgRNAs first, then significant sgRNAs on top
    ns  = sub[~sub["sig_either"]]
    sig = sub[sub["sig_either"]]
    ax.scatter(ns[xcol], ns[ycol], c="grey", s=8, alpha=0.25, edgecolors="none", zorder=1)
    ax.scatter(sig[xcol], sig[ycol], c="red", s=10, alpha=0.55, edgecolors="none", zorder=2)

    #y = x reference line, and lines at 0
    lims = [np.nanmin(np.concatenate([x, y])), np.nanmax(np.concatenate([x, y]))]
    ax.plot(lims, lims, "k--", lw=1)
    ax.axhline(0, color="black", lw=0.7)
    ax.axvline(0, color="black", lw=0.7)
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    ax.set_xlabel("Low FLZ log2FC", fontsize=11, fontweight="bold")
    ax.set_ylabel("High FLZ log2FC", fontsize=11, fontweight="bold")
    ax.set_title(f"{title}\nLow vs High FLZ sgRNA log2FC", fontsize=12, fontweight="bold", pad=12)
    ax.tick_params(axis="both", labelsize=10)
    for lab in ax.get_xticklabels() + ax.get_yticklabels():
        lab.set_fontweight("bold")

    ax.text(0.02, 0.98, f"Pearson r = {pr:.3f}\nSpearman ρ = {sr:.3f}",
            transform=ax.transAxes, ha="left", va="top", fontsize=10.5, fontweight="bold")

    n_sig = int(sub["sig_either"].sum())
    leg = ax.legend(
        handles=[Patch(color="red",  label=f"Significant sgRNAs (n={n_sig})"),
                 Patch(color="grey", label=f"Not significant (n={len(sub) - n_sig})")],
        frameon=False, fontsize=9, loc="lower right",
    )
    for text in leg.get_texts():
        text.set_fontweight("bold")

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"{out_prefix}_low_vs_high_scatter.png"), dpi=300, bbox_inches="tight")
    plt.close()

plot_low_vs_high("a", CI_A, "CRISPRa", "crisprA")
plot_low_vs_high("i", CI_I, "CRISPRi", "crisprI")

This checks how well the replicates correlate within each condition and screen, using Pearson correlations of TP3 sgRNA frequencies. Non-targeting sgRNAs are kept in since this is a QC check.

In [5]:
OUT_DIR = "./plotting/replicate_correlations"
os.makedirs(OUT_DIR, exist_ok=True)

conditions = ["YPD", "Low_FLZ", "High_FLZ"]
systems    = {"a": "CRISPRa", "i": "CRISPRi"}
reps       = [1, 2, 3, 4]

#Collect the frequency columns in order: CRISPRa (YPD, Low, High), then CRISPRi (YPD, Low, High)
cols, labels, block_ends = [], [], []
for suffix, system_name in systems.items():
    for cond in conditions:
        cols.extend(f"freq_SC5314_TP3_{cond}_{r}_{suffix}" for r in reps)
        labels.extend(f"{system_name} {cond} R{r}" for r in reps)
        block_ends.append(len(cols))   #index where this 4-replicate block ends

corr = df_all[cols].corr(method="pearson")
n = corr.shape[0]

fig, ax = plt.subplots(figsize=(6, 6), constrained_layout=True)
im = ax.imshow(corr.to_numpy(), vmin=0, vmax=1, cmap="Reds", aspect="equal")

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=90, fontsize=8, fontweight="bold")
ax.set_yticklabels(labels, fontsize=8, fontweight="bold")
ax.tick_params(axis="both", pad=1, length=3, width=1)

#Lines around each 4-replicate block
for b in block_ends:
    ax.axhline(b - 0.5, color="black", lw=2.0)
    ax.axvline(b - 0.5, color="black", lw=2.0)

#Thicker line between CRISPRa and CRISPRi (3 conditions x 4 replicates = 12)
ax.axhline(12 - 0.5, color="black", lw=3.0)
ax.axvline(12 - 0.5, color="black", lw=3.0)

for spine in ax.spines.values():
    spine.set_linewidth(2)

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Pearson r", fontsize=14, fontweight="bold", rotation=90, labelpad=12)
cbar.ax.tick_params(labelsize=10, width=1.2, length=3)
cbar.outline.set_linewidth(2)

fig.savefig(os.path.join(OUT_DIR, "TP3_freq_ALL_vs_ALL_correlation_matrix.png"), dpi=300, bbox_inches="tight")
plt.close(fig)

This now compares the overall similarity between the different conditions and screens, using Spearman correlations of sgRNA log2 fold changes. Only sgRNAs with values in all six conditions are used, and non-targeting sgRNAs are kept in.

In [6]:
OUT_DIR = "./plotting/screen_similarity"
os.makedirs(OUT_DIR, exist_ok=True)

vectors = {
    "CRISPRa YPD":      "log2_fold_change_ypd_a",
    "CRISPRa Low FLZ":  "log2_fold_change_low_flz_a",
    "CRISPRa High FLZ": "log2_fold_change_high_flz_a",
    "CRISPRi YPD":      "log2_fold_change_ypd_i",
    "CRISPRi Low FLZ":  "log2_fold_change_low_flz_i",
    "CRISPRi High FLZ": "log2_fold_change_high_flz_i",
}

#Rows = sgRNAs, columns = conditions. dropna() keeps only sgRNAs present in all six
X = pd.DataFrame({name: df_all[col] for name, col in vectors.items()}).dropna()
corr = X.corr(method="spearman")

fig, ax = plt.subplots(figsize=(5.8, 5.2), constrained_layout=True)
im = ax.imshow(corr.to_numpy(), vmin=0, vmax=1, cmap="Reds", aspect="equal")

labels = corr.columns.tolist()
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontweight="bold", fontsize=11)
ax.set_yticklabels(labels, fontweight="bold", fontsize=11)

#Write the correlation value in each cell
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontweight="bold", fontsize=12)

for spine in ax.spines.values():
    spine.set_linewidth(2)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Spearman Correlation", fontweight="bold", fontsize=12, labelpad=10)
cbar.ax.tick_params(labelsize=10, width=1.2, length=3)
cbar.outline.set_linewidth(2)

fig.savefig(os.path.join(OUT_DIR, "screen_similarity_log2FC_spearman.png"), dpi=300, bbox_inches="tight")
plt.close(fig)